<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 04. Deep Compression

## Deep Compression: Compressing Deep Neural Networks with Pruning, Trained Quantization and Huffman Coding

- **원 논문:** [Deep Compression: Compressing Deep Neural Networks with Pruning, Trained Quantization and Huffman Coding](https://arxiv.org/abs/1510.00149)
- **저자 / 발표:** Song Han, Huizi Mao, William J. Dally · ICLR 2016 (2016)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** magnitude pruning→weight-sharing quantization→entropy coding의 세 단계를 작은 weight tensor에 적용한다.

**축소하거나 생략한 부분:** 원 논문은 AlexNet/VGG를 재학습하고 실제 저장·에너지를 측정했다. 실습은 압축 지표와 reconstruction error만 계산한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 1: three-stage pipeline | Task 1–3 | 단계별 크기 |
| §3: network pruning | Task 1 | sparsity/mask |
| §4, Figure 2: trained quantization | Task 2 | 고정 assignment 아래 centroid gradient 재학습 |
| §5: Huffman coding | Task 3 | entropy lower-bound bit estimate |

## 미니 재현
프루닝은 0을 늘리고, 양자화는 남은 값을 centroid index로 바꾸며, entropy coding은 자주 나오는 index에 짧은 코드를 줍니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$\hat W=\operatorname{Huffman}(\operatorname{Quantize}(W\odot\mathbf1_{|W|>\tau}))$$

- **기호 정의:** τ는 pruning 임계값, Quantize는 centroid 공유, Huffman은 entropy coding입니다.
- **수식의 역할:** 0 제거, weight sharing, 빈도 기반 부호화를 순서대로 적용해 저장량을 줄입니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  Figure 1: three-stage pipeline의 흐름을 명시합니다.
- **입출력 shape:** dense weight [64, 32] → sparse mask → centroid index → 추정 bit 수
- **평가:** sparsity, centroid 재학습 전후 loss, entropy 기반 압축률
- **원문 대비 한계:** 원 논문은 AlexNet/VGG를 재학습하고 실제 저장·에너지를 측정했다. 실습은 압축 지표와 reconstruction error만 계산한다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3: network pruning
- **이 셀의 책임:** Task 1
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** sparsity/mask. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def magnitude_prune(weight, sparsity):
    threshold = torch.quantile(weight.abs().flatten(), sparsity)
    mask = weight.abs() > threshold
    return weight * mask, mask


pruned, mask = magnitude_prune(dense_weights, 0.70)
actual_sparsity = (~mask).float().mean().item()
assert 0.68 < actual_sparsity < 0.72 and torch.equal(pruned == 0, ~mask)

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §4, Figure 2: trained quantization
- **이 셀의 책임:** Task 2
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** 고정 assignment 아래 centroid gradient 재학습. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
values = pruned[mask]
centroids = torch.linspace(values.min(), values.max(), 8)
for _ in range(8):
    assignment = (values[:, None] - centroids[None, :]).abs().argmin(1)
    centroids = torch.stack(
        [
            values[assignment == i].mean() if (assignment == i).any() else centroids[i]
            for i in range(8)
        ]
    )
quantized = pruned.clone()
quantized[mask] = centroids[assignment]
quant_error = F.mse_loss(quantized, dense_weights).item()
assert (
    len(torch.unique(quantized)) <= 9
    and quant_error < dense_weights.square().mean().item()
)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §5: Huffman coding
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** entropy lower-bound bit estimate. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
symbols = torch.zeros_like(pruned, dtype=torch.long)
symbols[mask] = assignment + 1
counts = torch.bincount(symbols.flatten(), minlength=9).float()
probabilities = counts / counts.sum()
entropy = (
    -(probabilities[probabilities > 0] * probabilities[probabilities > 0].log2())
    .sum()
    .item()
)
estimated_bits = entropy * symbols.numel() + 32 * len(centroids)
original_bits = 32 * dense_weights.numel()
compression = original_bits / estimated_bits
assert entropy < math.log2(9) and compression > 4
plt.bar(
    ["dense", "pruned", "shared"],
    [dense_weights.numel(), int(mask.sum()), len(torch.unique(quantized))],
)
plt.ylabel("stored values / symbols")
plt.show()
print(
    f"sparsity={actual_sparsity:.1%}, entropy={entropy:.2f} bit, "
    f"estimated compression={compression:.1f}x"
)

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Figure 1: three-stage pipeline / §3: network pruning / §4, Figure 2: trained quantization / §5: Huffman coding
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 단계별 크기 / sparsity/mask / 고정 assignment 아래 centroid gradient 재학습 / entropy lower-bound bit estimate. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    keep_ratio: float = 0.3
    cluster_count: int = 6
    learning_rate: float = 0.025
    steps: int = 16
    centroid_steps: int = 8


def prepare_batch() -> tuple[Tensor, Tensor]:
    return features[train_idx], labels[train_idx]


class DeepCompressionPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.linear = nn.Linear(features.shape[1], 48)
        self.classifier = nn.Linear(48, teacher_logits.shape[1])
        self.register_buffer("mask", torch.ones_like(self.linear.weight))
        self.centroids = nn.Parameter(torch.zeros(config.cluster_count))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.linear.weight, dtype=torch.long),
        )
        self.register_buffer("sharing_active", torch.tensor(False))
        self.pruned_train_loss = float("nan")
        self.quantized_loss_before = float("nan")
        self.quantized_loss_after = float("nan")
        self.centroid_update_norm = 0.0
        self.sharing_optimizer_fresh = 0

    def effective_weight(self) -> Tensor:
        if bool(self.sharing_active):
            shared = self.centroids[self.assignments]
            return shared * self.mask
        return self.linear.weight * self.mask

    def forward(self, inputs: Tensor) -> Tensor:
        hidden = F.relu(F.linear(inputs, self.effective_weight(), self.linear.bias))
        return self.classifier(hidden)

    def compute_loss(self, logits: Tensor, targets: Tensor) -> Tensor:
        return F.cross_entropy(logits, targets)

    def training_step(self, inputs: Tensor, targets: Tensor) -> Tensor:
        return self.compute_loss(self(inputs), targets)

    def prune(self) -> None:
        with torch.no_grad():
            threshold = torch.quantile(
                self.linear.weight.abs(),
                1.0 - self.config.keep_ratio,
            )
            self.mask.copy_((self.linear.weight.abs() >= threshold).float())
            self.linear.weight.mul_(self.mask)

    def apply_weight_sharing(self) -> None:
        with torch.no_grad():
            survivor = self.linear.weight[self.mask.bool()]
            fractions = torch.linspace(0.0, 1.0, self.config.cluster_count)
            centroids = torch.quantile(survivor, fractions.to(survivor.device))
            for _ in range(6):
                distance = (survivor[:, None] - centroids[None, :]).abs()
                survivor_assignment = distance.argmin(dim=1)
                for index in range(self.config.cluster_count):
                    members = survivor[survivor_assignment == index]
                    if members.numel():
                        centroids[index] = members.mean()
            full_assignment = torch.zeros_like(self.assignments)
            distance = (survivor[:, None] - centroids[None, :]).abs()
            full_assignment[self.mask.bool()] = distance.argmin(dim=1)
            self.centroids.copy_(centroids)
            self.assignments.copy_(full_assignment)
            self.sharing_active.fill_(True)

    def coding_proxy(self) -> dict[str, float]:
        symbols = self.assignments[self.mask.bool()]
        counts = torch.bincount(symbols, minlength=self.config.cluster_count).float()
        probability = counts[counts > 0] / counts.sum().clamp_min(1.0)
        entropy = -(probability * probability.log2()).sum()
        index_bits = entropy * symbols.numel()
        compressed_bits = (
            index_bits + 32 * self.config.cluster_count + self.mask.numel()
        )
        dense_bits = 32 * self.mask.numel()
        return {
            "index_entropy_bits": float(entropy.cpu()),
            "estimated_compression_ratio": float((dense_bits / compressed_bits).cpu()),
        }

    def enforce_mask(self) -> None:
        with torch.no_grad():
            self.linear.weight.mul_(self.mask)

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** Figure 1: three-stage pipeline / §3: network pruning / §4, Figure 2: trained quantization / §5: Huffman coding
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 단계별 크기 / sparsity/mask / 고정 assignment 아래 centroid gradient 재학습 / entropy lower-bound bit estimate. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: DeepCompressionPortfolioModel) -> list[float]:
    inputs, targets = prepare_batch()
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    warmup = max(4, model.config.steps // 2)
    history = []
    for step in range(model.config.steps):
        loss = model.training_step(inputs, targets)
        optimizer.zero_grad()
        loss.backward()
        if model.linear.weight.grad is not None:
            model.linear.weight.grad.mul_(model.mask)
        optimizer.step()
        model.enforce_mask()
        history.append(float(loss.detach().cpu()))
        if step + 1 == warmup:
            model.prune()
    with torch.no_grad():
        model.pruned_train_loss = float(
            model.training_step(inputs, targets).detach().cpu()
        )
    model.apply_weight_sharing()
    with torch.no_grad():
        model.quantized_loss_before = float(
            model.training_step(inputs, targets).detach().cpu()
        )
    centroids_before = model.centroids.detach().clone()
    sharing_parameters = [
        model.centroids,
        model.linear.bias,
        *model.classifier.parameters(),
    ]
    sharing_optimizer = torch.optim.Adam(
        sharing_parameters,
        lr=model.config.learning_rate,
    )
    model.sharing_optimizer_fresh = int(len(sharing_optimizer.state) == 0)
    for _ in range(model.config.centroid_steps):
        loss = model.training_step(inputs, targets)
        sharing_optimizer.zero_grad()
        loss.backward()
        sharing_optimizer.step()
        history.append(float(loss.detach().cpu()))
    with torch.no_grad():
        model.quantized_loss_after = float(
            model.training_step(inputs, targets).detach().cpu()
        )
        model.centroid_update_norm = float(
            (model.centroids - centroids_before).norm().cpu()
        )
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** Figure 1: three-stage pipeline / §3: network pruning / §4, Figure 2: trained quantization / §5: Huffman coding
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** weight [O,I] → sparse/quantized symbols [O,I] + compression ratio scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 단계별 크기 / sparsity/mask / 고정 assignment 아래 centroid gradient 재학습 / entropy lower-bound bit estimate. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(model: DeepCompressionPortfolioModel) -> dict[str, float]:
    with torch.no_grad():
        logits = model(features[test_idx])
        accuracy = (logits.argmax(1) == labels[test_idx]).float().mean()
        sparsity = 1.0 - model.mask.mean()
        shared_values = torch.unique(
            model.effective_weight()[model.mask.bool()]
        ).numel()
    metrics = {
        "test_accuracy": float(accuracy.cpu()),
        "weight_sparsity": float(sparsity.cpu()),
        "shared_weight_values": float(shared_values),
        "pruned_train_loss": model.pruned_train_loss,
        "quantized_loss_before_retraining": model.quantized_loss_before,
        "quantized_loss_after_retraining": model.quantized_loss_after,
        "centroid_update_norm": model.centroid_update_norm,
        "centroid_retraining_steps": float(model.config.centroid_steps),
        "sharing_optimizer_fresh": float(model.sharing_optimizer_fresh),
    }
    metrics.update(model.coding_proxy())
    return metrics


portfolio_model = DeepCompressionPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["centroid_update_norm"] > 0.0
assert (
    portfolio_metrics["quantized_loss_after_retraining"]
    < portfolio_metrics["quantized_loss_before_retraining"]
)
assert portfolio_metrics["estimated_compression_ratio"] > 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 AlexNet/VGG를 재학습하고 실제 저장·에너지를 측정했다. 실습은 압축 지표와 reconstruction error만 계산한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.